# Fine-tuning GLiNER2 on OntoNotes5 (PERSON, ORG, GPE, EVENT, MONEY)

This notebook fine-tunes **GLiNER2** (`fastino/gliner2-base-v1`) on the [`tner/ontonotes5`](https://huggingface.co/datasets/tner/ontonotes5) dataset, restricted to 5 entity types:

- `PERSON`
- `ORG`
- `GPE`
- `EVENT`
- `MONEY`

**Runtime:** Go to `Runtime > Change runtime type > T4 GPU` before running this notebook.

**Pipeline overview:**
1. Install dependencies
2. Load `tner/ontonotes5` and convert its BIO tags into GLiNER2's `InputExample` format, keeping only the 5 target labels
3. Build train / validation `TrainingDataset`s
4. Load the pretrained `gliner2-base-v1` model and fine-tune it with `GLiNER2Trainer`
5. Evaluate precision / recall / F1 per label on the test split
6. Run a few sanity-check predictions
7. Zip and download the fine-tuned model


## 1. Check GPU

In [ ]:
!nvidia-smi


Thu Aug 13 13:28:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install dependencies

In [ ]:
# gliner2[local] pulls in torch/transformers for local training + inference
!pip install -q "gliner2[local]" datasets seqeval


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.8/96.8 kB 9.9 MB/s eta 0:00:00


In [ ]:
import os
import json
import random
from collections import Counter, defaultdict

import torch
from datasets import load_dataset

from gliner2 import GLiNER2
from gliner2.training.data import InputExample, TrainingDataset
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)


Using device: cuda


## 3. Load `tner/ontonotes5`

The dataset stores each sentence as `tokens` (list of words) and `tags` (list of integer BIO tag ids).
The id -> tag mapping is published in the dataset's `label.json` and is hard-coded below so we don't
depend on the `datasets` library exposing a `ClassLabel` feature.

In [ ]:
from datasets import load_dataset

# Load from the auto-converted Parquet export (no custom loading script,
# no trust_remote_code needed, and it's faster to load).
raw = load_dataset("tner/ontonotes5", revision="refs/convert/parquet")
print(raw)

ontonotes5/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 3.68MB            

ontonotes5/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/515k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/517k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 59924
    })
    validation: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8528
    })
    test: Dataset({
        features: ['tokens', 'tags'],
        num_rows: 8262
    })
})


In [ ]:
# Official label2id mapping for tner/ontonotes5
# https://huggingface.co/datasets/tner/ontonotes5/raw/main/dataset/label.json
LABEL2ID = {
    "O": 0,
    "B-CARDINAL": 1, "B-DATE": 2, "I-DATE": 3, "B-PERSON": 4, "I-PERSON": 5,
    "B-NORP": 6, "B-GPE": 7, "I-GPE": 8, "B-LAW": 9, "I-LAW": 10,
    "B-ORG": 11, "I-ORG": 12, "B-PERCENT": 13, "I-PERCENT": 14, "B-ORDINAL": 15,
    "B-MONEY": 16, "I-MONEY": 17, "B-WORK_OF_ART": 18, "I-WORK_OF_ART": 19, "B-FAC": 20,
    "B-TIME": 21, "I-CARDINAL": 22, "B-LOC": 23, "B-QUANTITY": 24, "I-QUANTITY": 25,
    "I-NORP": 26, "I-LOC": 27, "B-PRODUCT": 28, "I-TIME": 29, "B-EVENT": 30,
    "I-EVENT": 31, "I-FAC": 32, "B-LANGUAGE": 33, "I-PRODUCT": 34, "I-ORDINAL": 35,
    "I-LANGUAGE": 36,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# The 5 entity types we want to fine-tune on. These match OntoNotes5 tag names exactly,
# so no renaming is needed.
TARGET_LABELS = ["PERSON", "ORG", "GPE", "EVENT", "MONEY"]

ENTITY_DESCRIPTIONS = {
    "PERSON": "Names of people, including fictional characters",
    "ORG": "Companies, agencies, institutions, and other organizations",
    "GPE": "Countries, cities, states, and other geo-political entities",
    "EVENT": "Named events such as wars, hurricanes, sports events, and named incidents",
    "MONEY": "Monetary values, including amounts with currency symbols or units",
}


## 4. Convert BIO tags to GLiNER2 `InputExample`s

For each sentence we:
1. Join tokens into a single whitespace-separated string (GLiNER2 tokenizes text itself, so this is fine, and it keeps the entity substrings we extract in exact alignment with the joined text).
2. Walk the BIO tags and collect contiguous `B-`/`I-` spans whose label is one of `TARGET_LABELS`.
3. Drop any tag that isn't one of our 5 target labels (e.g. `DATE`, `CARDINAL`, `NORP`, ...) — those tokens are simply treated as non-entity text.
4. Keep sentences with **no** target entities too (negative examples), but downsample them so the training set isn't overwhelmingly negative — most OntoNotes5 sentences don't contain a PERSON/ORG/GPE/EVENT/MONEY mention.

In [ ]:
def bio_to_entities(tokens, tag_ids, id2label, target_labels):
    """Convert a BIO-tagged sentence into {label: [entity text, ...]}."""
    entities = defaultdict(list)
    cur_label, cur_tokens = None, []

    def flush():
        nonlocal cur_label, cur_tokens
        if cur_label is not None and cur_tokens:
            entities[cur_label].append(" ".join(cur_tokens))
        cur_label, cur_tokens = None, []

    for tok, tid in zip(tokens, tag_ids):
        tag = id2label.get(tid, "O")
        if tag == "O":
            flush()
            continue
        prefix, _, label = tag.partition("-")
        if label not in target_labels:
            flush()
            continue
        if prefix == "B" or cur_label != label:
            flush()
            cur_label, cur_tokens = label, [tok]
        else:  # "I-" continuing the same label
            cur_tokens.append(tok)
    flush()
    return dict(entities)


def build_examples(split, target_labels, entity_descriptions, negative_keep_ratio=1.0, seed=SEED):
    """Turn a HF dataset split into a list of gliner2 InputExample.

    negative_keep_ratio: fraction of sentences with ZERO target entities to keep
                          (set < 1.0 to reduce the negative/positive imbalance).
    """
    rng = random.Random(seed)
    examples = []
    n_pos, n_neg_total, n_neg_kept = 0, 0, 0

    for tokens, tags in zip(split["tokens"], split["tags"]):
        if not tokens:
            continue
        text = " ".join(tokens)
        entities = bio_to_entities(tokens, tags, ID2LABEL, target_labels)

        if entities:
            n_pos += 1
        else:
            n_neg_total += 1
            if rng.random() > negative_keep_ratio:
                continue
            n_neg_kept += 1

        examples.append(
            InputExample(
                text=text,
                entities=entities,
                entity_descriptions=entity_descriptions,
            )
        )

    print(
        f"positives={n_pos}, negatives kept={n_neg_kept}/{n_neg_total}, "
        f"total examples={len(examples)}"
    )
    return examples


In [ ]:
# Keep every sentence that has at least one target entity, and ~25% of the
# entity-free sentences, so the model still sees plenty of "no entity" text
# without training being dominated by it. Tune this if you want faster/slower runs.
NEGATIVE_KEEP_RATIO = 0.25

# Optional: cap dataset size for a quick smoke-test run on Colab.
# Set to None to use the full splits.
SUBSET_SIZE = None  # e.g. 8000 for a fast first pass

train_split = raw["train"] if SUBSET_SIZE is None else raw["train"].select(range(min(SUBSET_SIZE, len(raw["train"]))))
val_split = raw["validation"]
test_split = raw["test"]

train_examples = build_examples(train_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=NEGATIVE_KEEP_RATIO)
val_examples = build_examples(val_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=NEGATIVE_KEEP_RATIO)
test_examples = build_examples(test_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=1.0)


positives=24702, negatives kept=8811/35222, total examples=33513
positives=3310, negatives kept=1276/5218, total examples=4586
positives=3374, negatives kept=4888/4888, total examples=8262


In [ ]:
# Sanity check a couple of converted examples
for ex in train_examples[:5]:
    if ex.entities:
        print(ex.text)
        print(ex.entities)
        print("-" * 60)


Last week , Sen. Malcolm Wallop -LRB- R. , Wyo . -RRB- held hearings on a bill to strengthen an existing law designed to reduce regulatory hassles for small businesses .
{'PERSON': ['Malcolm Wallop'], 'GPE': ['Wyo .']}
------------------------------------------------------------
`` A great many federal regulations are meant for larger entities and do n't really apply to small businesses , '' says Marian Jacob , a legislative aide to Sen. Wallop .
{'PERSON': ['Marian Jacob', 'Wallop']}
------------------------------------------------------------
Thus , optimistic entrepreneurs await a promised land of less red tape -- just as soon as Uncle Sam gets around to arranging it .
{'PERSON': ['Uncle Sam']}
------------------------------------------------------------


In [ ]:
# Per-label mention counts in the training set
label_counts = Counter()
for ex in train_examples:
    for label, mentions in ex.entities.items():
        label_counts[label] += len(mentions)
print("Training set entity mention counts:")
for label in TARGET_LABELS:
    print(f"  {label}: {label_counts[label]}")


Training set entity mention counts:
  PERSON: 15429
  ORG: 12820
  GPE: 15405
  EVENT: 748
  MONEY: 2434


## 5. Validate the dataset with GLiNER2's `TrainingDataset`

In [ ]:
def bio_to_entities(tokens, tag_ids, id2label, target_labels):
    """Convert a BIO-tagged sentence into {label: [entity text, ...]}."""
    entities = defaultdict(list)
    cur_label, cur_tokens = None, []

    def flush():
        nonlocal cur_label, cur_tokens
        if cur_label is not None and cur_tokens:
            entities[cur_label].append(" ".join(cur_tokens))
        cur_label, cur_tokens = None, []

    for tok, tid in zip(tokens, tag_ids):
        tag = id2label.get(tid, "O")
        if tag == "O":
            flush()
            continue
        prefix, _, label = tag.partition("-")
        if label not in target_labels:
            flush()
            continue
        if prefix == "B" or cur_label != label:
            flush()
            cur_label, cur_tokens = label, [tok]
        else:  # "I-" continuing the same label
            cur_tokens.append(tok)
    flush()
    return dict(entities)


def build_examples(split, target_labels, entity_descriptions, negative_keep_ratio=1.0, seed=SEED):
    """Turn a HF dataset split into a list of gliner2 InputExample.

    negative_keep_ratio: fraction of sentences with ZERO target entities to keep
                          (set < 1.0 to reduce the negative/positive imbalance).
    """
    rng = random.Random(seed)
    examples = []
    n_pos, n_neg_total, n_neg_kept = 0, 0, 0

    for tokens, tags in zip(split["tokens"], split["tags"]):
        if not tokens:
            continue
        text = " ".join(tokens)
        found = bio_to_entities(tokens, tags, ID2LABEL, target_labels)
        is_positive = bool(found)

        if is_positive:
            n_pos += 1
        else:
            n_neg_total += 1
            if rng.random() > negative_keep_ratio:
                continue
            n_neg_kept += 1

        # entities dict must contain EVERY target label as a key (even if empty),
        # since entity_descriptions keys must match entities keys exactly, and a
        # non-empty entities dict is what makes an example count as "having content".
        entities = {label: found.get(label, []) for label in target_labels}

        examples.append(
            InputExample(
                text=text,
                entities=entities,
                entity_descriptions=entity_descriptions,
            )
        )

    print(
        f"positives={n_pos}, negatives kept={n_neg_kept}/{n_neg_total}, "
        f"total examples={len(examples)}"
    )
    return examples

In [ ]:
train_examples = build_examples(train_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=NEGATIVE_KEEP_RATIO)
val_examples = build_examples(val_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=NEGATIVE_KEEP_RATIO)
test_examples = build_examples(test_split, TARGET_LABELS, ENTITY_DESCRIPTIONS, negative_keep_ratio=1.0)

positives=24702, negatives kept=8811/35222, total examples=33513
positives=3310, negatives kept=1276/5218, total examples=4586
positives=3374, negatives kept=4888/4888, total examples=8262


In [ ]:
train_dataset = TrainingDataset(train_examples)
train_dataset.validate(raise_on_error=True)
train_dataset.print_stats()

val_dataset = TrainingDataset(val_examples)
val_dataset.validate(raise_on_error=True)
val_dataset.print_stats()


GLiNER2 Training Dataset Statistics
Total examples: 33513

Text lengths: min=1, max=1398, mean=116.8

Task Distribution:
  entities_only: 33513 (100.0%)

Entity Types (46836 total mentions):
  PERSON: 15429
  GPE: 15405
  ORG: 12820
  MONEY: 2434
  EVENT: 748


GLiNER2 Training Dataset Statistics
Total examples: 4586

Text lengths: min=1, max=1179, mean=113.7

Task Distribution:
  entities_only: 4586 (100.0%)

Entity Types (6445 total mentions):
  GPE: 2268
  PERSON: 2020
  ORG: 1740
  MONEY: 274
  EVENT: 143



## 6. Load the pretrained model and configure training

`gliner2-base-v1` (205M params) is a good default for Colab. Swap in `gliner2-large-v1` if you have
a bigger GPU and more time.

The settings below (batch size 16, fp16, cosine schedule, early stopping) are reasonable defaults for
a single Colab T4. Lower `batch_size` if you hit an out-of-memory error.

In [ ]:
model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
model.to(DEVICE)

config = TrainingConfig(
    output_dir="./gliner2_ontonotes5_ner",
    experiment_name="ontonotes5_person_org_gpe_event_money",
    num_epochs=6,
    batch_size=16,
    encoder_lr=1e-5,
    task_lr=5e-4,
    warmup_ratio=0.1,
    scheduler_type="cosine",
    fp16=(DEVICE == "cuda"),
    eval_strategy="epoch",
    save_best=True,
    early_stopping=True,
    early_stopping_patience=3,
)


config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/823 [00:00<?, ?B/s]

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/3.24k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

...

added_tokens.json:   0%|          | 0.00/230 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.41k [00:00<?, ?B/s]

🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first


model.safetensors: reconstructing file:   0%|          |  0.00B /  834MB            

model.safetensors: downloading bytes:           |  0.00B            

## 7. Train

In [ ]:
trainer = GLiNER2Trainer(model, config)
trainer.train(train_data=train_examples, eval_data=val_examples)

Validating records: 100%|██████████| 33513/33513 [00:00<00:00, 108548.18record/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:876: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)


Training:   0%|          | 0/12564 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:927: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:973: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:1096: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):


...

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

{'total_steps': 12564,
 'total_epochs': 6,
 'total_time_seconds': 6205.246208429337,
 'samples_per_second': 32.39581367890363,
 'best_metric': 9.299130061118886,
 'train_metrics_history': [{'loss': 56.21823501586914,
   'classification_loss': 0.0,
   'structure_loss': 56.21823501586914,
   'count_loss': 0.0,
   'learning_rate': 7.961783439490447e-09,
   'epoch': 0.0,
   'step': 1,
   'samples_seen': 16,
   'throughput': 6.091029715488459},
  {'loss': 80.91007995605469,
   'classification_loss': 0.0,
   'structure_loss': 80.91007995605469,
   'count_loss': 0.0,
   'learning_rate': 1.5923566878980894e-08,
   'epoch': 0.0004775549188156638,
   'step': 2,
   'samples_seen': 32,
   'throughput': 9.643926580180741},
  {'loss': 139.92568969726562,
   'classification_loss': 0.0,
   'structure_loss': 139.92568969726562,
   'count_loss': 0.0,
   'learning_rate': 2.388535031847134e-08,
   'epoch': 0.0009551098376313276,
   'step': 3,
   'samples_seen': 48,
   'throughput': 12.449355316168148},
  

In [ ]:
# Reload the best checkpoint saved during training
best_model_path = os.path.join(config.output_dir, "best")
model = GLiNER2.from_pretrained(best_model_path)
model.to(DEVICE)
print("Loaded best checkpoint from", best_model_path)


[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Loaded best checkpoint from ./gliner2_ontonotes5_ner/best


## 8. Evaluate on the held-out test split

We run entity extraction on every test sentence restricted to our 5 labels, and compare the
predicted mention strings against the gold mentions (exact string match, per label) to compute
precision / recall / F1.

In [ ]:
def evaluate(model, examples, target_labels, batch_size=32):
    texts = [ex.text for ex in examples]
    gold_all = [ex.entities for ex in examples]

    preds_all = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        batch_preds = model.batch_extract_entities(batch, target_labels, batch_size=batch_size)
        preds_all.extend(batch_preds)

    stats = {label: {"tp": 0, "fp": 0, "fn": 0} for label in target_labels}

    for gold, pred in zip(gold_all, preds_all):
        pred_entities = pred.get("entities", pred) if isinstance(pred, dict) else {}
        for label in target_labels:
            gold_set = set(gold.get(label, []))
            pred_set = set(pred_entities.get(label, []))
            stats[label]["tp"] += len(gold_set & pred_set)
            stats[label]["fp"] += len(pred_set - gold_set)
            stats[label]["fn"] += len(gold_set - pred_set)

    print(f"{'label':<10}{'precision':>10}{'recall':>10}{'f1':>10}{'support':>10}")
    macro_f1 = []
    for label in target_labels:
        tp, fp, fn = stats[label]["tp"], stats[label]["fp"], stats[label]["fn"]
        precision = tp / (tp + fp) if (tp + fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
        macro_f1.append(f1)
        support = tp + fn
        print(f"{label:<10}{precision:>10.3f}{recall:>10.3f}{f1:>10.3f}{support:>10}")
    print(f"\nmacro F1: {sum(macro_f1) / len(macro_f1):.3f}")
    return stats


eval_stats = evaluate(model, test_examples, TARGET_LABELS)


## 9. Quick sanity-check predictions

In [ ]:
sample_texts = [
  "Beginning on 28 February 2026, the United States and Israel launched a war against Iran and its regional allies, initiated by airstrikes that killed Iranian Supreme Leader Ali Khamenei during ongoing nuclear negotiations. The conflict expanded significantly as Iranian forces retaliated with drone and missile strikes, blockaded and annexed the Strait of Hormuz, and reignited fighting with Hezbollah. In response to regional strikes, Saudi Arabia, the UAE, and Kuwait joined offensive operations against Iran, while Israel began occupying southern Lebanon. Spurring massive global fallout, the war caused high casualties, severe volatility across financial markets, and the largest oil supply disruption in history. By mid-2026, the ongoing conflict had cost the U.S. over $113 billion and prompted a record-setting $1.5 trillion Pentagon budget request from the Trump administration."]

for text in sample_texts:
    result = model.extract_entities(text, TARGET_LABELS)
    print(text)
    print(result)
    print("-" * 80)


Beginning on 28 February 2026, the United States and Israel launched a war against Iran and its regional allies, initiated by airstrikes that killed Iranian Supreme Leader Ali Khamenei during ongoing nuclear negotiations. The conflict expanded significantly as Iranian forces retaliated with drone and missile strikes, blockaded and annexed the Strait of Hormuz, and reignited fighting with Hezbollah. In response to regional strikes, Saudi Arabia, the UAE, and Kuwait joined offensive operations against Iran, while Israel began occupying southern Lebanon. Spurring massive global fallout, the war caused high casualties, severe volatility across financial markets, and the largest oil supply disruption in history. By mid-2026, the ongoing conflict had cost the U.S. over $113 billion and prompted a record-setting $1.5 trillion Pentagon budget request from the Trump administration.
{'entities': {'PERSON': ['Ali Khamenei', 'Trump'], 'ORG': ['Pentagon', 'Hezbollah'], 'GPE': ['Israel', 'Kuwait', '

## 10. Save and download the fine-tuned model

This zips the best checkpoint directory so you can download it from Colab, or optionally save it
straight to Google Drive if you'd rather persist it there.

In [ ]:
import shutil

zip_path = shutil.make_archive("gliner2_ontonotes5_ner_best", "zip", best_model_path)
print("Zipped model to:", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — the zip file is at:", zip_path)


In [ ]:
# Reload the best checkpoint from the first fine‑tuning run
best_model_path = "./gliner2_ontonotes5_ner/best"   # adjust if you saved elsewhere
model = GLiNER2.from_pretrained(best_model_path)
model.to(DEVICE)
print("Loaded model from", best_model_path)

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Loaded model from ./gliner2_ontonotes5_ner/best


In [ ]:
# ============================================================
#  UPLOAD AND PARSE YOUR combin_out DATASET (JSONL)
#  Format: {"tokenized_text": [...], "ner": [[start, end, label], ...]}
# ============================================================
from google.colab import files
import json
from gliner2.training.data import InputExample

# Set to True to keep only EVENT entities from this file (recommended to boost EVENT)
KEEP_ONLY_EVENT = True   # Change to False if you want all labels from this file

def load_uploaded_jsonl(file_content, entity_descriptions, keep_only_event=False):
    examples = []
    text_data = file_content.decode('utf-8')
    for line in text_data.splitlines():
        line = line.strip()
        if not line:
            continue
        data = json.loads(line)
        tokens = data.get("tokenized_text", [])
        if not tokens:
            continue

        # Reconstruct text by joining tokens with spaces
        text = " ".join(tokens)

        ner_list = data.get("ner", [])
        # Build a dict with only the labels we care about (or all if not filtering)
        if keep_only_event:
            entities = {label: [] for label in TARGET_LABELS}
            # Find all EVENT mentions
            event_mentions = []
            for start, end, label in ner_list:
                if label == "EVENT":  # only take EVENT
                    # Extract the span of tokens from start to end inclusive
                    mention = " ".join(tokens[start:end+1])
                    event_mentions.append(mention)
            if not event_mentions:
                continue  # skip if no EVENT in this example
            entities["EVENT"] = event_mentions
        else:
            # Keep all labels as they appear, but only those in TARGET_LABELS (or all)
            entities = {}
            for start, end, label in ner_list:
                if label not in entities:
                    entities[label] = []
                mention = " ".join(tokens[start:end+1])
                entities[label].append(mention)
            # Optionally filter to only TARGET_LABELS
            # entities = {k: v for k, v in entities.items() if k in TARGET_LABELS}

        examples.append(
            InputExample(
                text=text,
                entities=entities,
                entity_descriptions=entity_descriptions
            )
        )
    return examples

# Upload the file – drag & drop or click to select
uploaded = files.upload()

if uploaded:
    filename, content = next(iter(uploaded.items()))
    print(f"Uploaded: {filename} ({len(content)} bytes)")
    event_examples = load_uploaded_jsonl(content, ENTITY_DESCRIPTIONS, keep_only_event=KEEP_ONLY_EVENT)
    print(f"Loaded {len(event_examples)} examples from the uploaded file.")

    # Quick preview
    for i, ex in enumerate(event_examples[:3]):
        print(f"\n--- Example {i+1} ---")
        print("Text:", ex.text[:150] + "..." if len(ex.text) > 150 else ex.text)
        print("Entities (only EVENT):", ex.entities)
else:
    print("No file uploaded.")

Saving combined_output.jsonl to combined_output (1).jsonl
Uploaded: combined_output (1).jsonl (7804778 bytes)
Loaded 2037 examples from the uploaded file.

--- Example 1 ---
Text: The tornado outbreak continues from Easter Sunday across the Southeastern United States . At least 30 people are confirmed dead . ( USA Today )
Entities (only EVENT): {'PERSON': [], 'ORG': [], 'GPE': [], 'EVENT': ['Easter Sunday'], 'MONEY': []}

--- Example 2 ---
Text: Cyclone Harold hits Fiji . ( UPI )
Entities (only EVENT): {'PERSON': [], 'ORG': [], 'GPE': [], 'EVENT': ['Cyclone Harold'], 'MONEY': []}

--- Example 3 ---
Text: YouTube says it will remove videos promoting a conspiracy theory linking 5G to COVID-19 , while " borderline content " will be removed from search res...
Entities (only EVENT): {'PERSON': [], 'ORG': [], 'GPE': [], 'EVENT': ['5G'], 'MONEY': []}


In [ ]:
# ============================================================
#  ANALYZE LABEL DISTRIBUTION IN THE UPLOADED FILE
# ============================================================
from collections import Counter

def analyze_labels(examples):
    label_counts = Counter()
    for ex in examples:
        for label, mentions in ex.entities.items():
            label_counts[label] += len(mentions)
    return label_counts

if 'event_examples' in locals() and event_examples:
    label_counts = analyze_labels(event_examples)
    print("Entity type counts in the uploaded file:")
    for label, count in sorted(label_counts.items()):
        print(f"  {label}: {count}")

    # Check if EVENT is present
    if "EVENT" in label_counts and label_counts["EVENT"] > 0:
        print(f"\n✅ Found {label_counts['EVENT']} EVENT mentions – good! You can proceed with the boost.")
    else:
        print("\n⚠️ WARNING: No EVENT entities found in the uploaded file.")
        print("   This file will NOT help improve EVENT detection.")
        print("   Options:")
        print("   1. Upload a different dataset that contains EVENT annotations.")
        print("   2. If your file has other entity types (e.g., ORG, PERSON), you can still combine them")
        print("      to fine‑tune the model on more data, but EVENT will not improve from this file.")
        print("   3. Manually annotate some examples with EVENT or use a synthetic approach.")
else:
    print("No examples loaded. Please upload the file again with the correct format.")

Entity type counts in the uploaded file:
  EVENT: 2288
  GPE: 0
  MONEY: 0
  ORG: 0
  PERSON: 0

✅ Found 2288 EVENT mentions – good! You can proceed with the boost.


In [ ]:
# ============================================================
#  CONTINUED FINE‑TUNING WITH EVENT‑RICH DATA (FIXED IMPORTS)
# ============================================================
import os
from gliner2.training.trainer import GLiNER2Trainer, TrainingConfig   # ✅ correct import
from gliner2.training.data import TrainingDataset                     # optional, for validation

# Ensure event_examples exists and is not empty
if 'event_examples' not in locals() or not event_examples:
    print("⚠️ No event examples found. Please upload a valid file first.")
    print("   If your file does not contain EVENT annotations, you can still")
    print("   combine all its labels by re‑uploading with KEEP_ONLY_EVENT = False.")
else:
    # Combine with original training data
    combined_train_examples = train_examples + event_examples
    print(f"Original training examples: {len(train_examples)}")
    print(f"Additional examples from uploaded file: {len(event_examples)}")
    print(f"Total combined training examples: {len(combined_train_examples)}")

    # Optional: validate the combined dataset
    combined_dataset = TrainingDataset(combined_train_examples)
    combined_dataset.validate(raise_on_error=True)
    combined_dataset.print_stats()

    # Configure continued training – lower LR, fewer epochs
    config_continued = TrainingConfig(
        output_dir="./gliner2_event_boosted",
        experiment_name="event_boost",
        num_epochs=3,                     # short fine‑tuning phase
        batch_size=16,
        encoder_lr=1e-6,                  # much lower than initial fine‑tuning
        task_lr=5e-5,
        warmup_ratio=0.1,
        scheduler_type="cosine",
        fp16=(DEVICE == "cuda"),
        eval_strategy="epoch",
        save_best=True,
        early_stopping=True,
        early_stopping_patience=3,
    )

    # Train
    trainer = GLiNER2Trainer(model, config_continued)
    trainer.train(train_data=combined_train_examples, eval_data=val_examples)

    # Reload the best checkpoint
    best_continued = os.path.join(config_continued.output_dir, "best")
    model = GLiNER2.from_pretrained(best_continued)
    model.to(DEVICE)
    print(f"✅ Loaded boosted model from {best_continued}")

    # Evaluate on the test set
    eval_stats = evaluate(model, test_examples, TARGET_LABELS)

Original training examples: 33513
Additional examples from uploaded file: 2037
Total combined training examples: 35550

GLiNER2 Training Dataset Statistics
Total examples: 35550

Text lengths: min=1, max=1398, mean=124.3

Task Distribution:
  entities_only: 35550 (100.0%)

Entity Types (49124 total mentions):
  PERSON: 15429
  GPE: 15405
  ORG: 12820
  EVENT: 3036
  MONEY: 2434



Validating records: 100%|██████████| 35550/35550 [00:00<00:00, 205332.46record/s]
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:876: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=self.config.fp16)


Training:   0%|          | 0/6663 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:927: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):
/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:973: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/gliner2/training/trainer.py:1096: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp, dtype=amp_dtype):


Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/574 [00:00<?, ?it/s]

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
✅ Loaded boosted model from ./gliner2_event_boosted/best
label      precision    recall        f1   support
PERSON         0.910     0.945     0.927      1961
ORG            0.815     0.850     0.832      1751
GPE            0.948     0.951     0.950      2142
EVENT          0.439     0.705     0.541        61
MONEY          0.882     0.907     0.894       313

macro F1: 0.829


In [ ]:
# ============================================================
#  EVALUATE AND DOWNLOAD THE FINAL MODEL
# ============================================================
import os
import shutil

# Check if continued training was performed and produced a checkpoint
best_continued = "./gliner2_event_boosted/best"
if os.path.exists(best_continued):
    print(f"✅ Loading continued fine‑tuned model from {best_continued}")
    model = GLiNER2.from_pretrained(best_continued)
    model.to(DEVICE)
    print("Model loaded from continued training.")
else:
    print("⚠️ No continued checkpoint found. Using the original best model.")
    original_best = "./gliner2_ontonotes5_ner/best"
    if os.path.exists(original_best):
        model = GLiNER2.from_pretrained(original_best)
        model.to(DEVICE)
        print("Model loaded from original fine‑tuning.")
    else:
        # Fallback to the base model if nothing else is found
        print("⚠️ No saved model found. Loading base gliner2 model.")
        model = GLiNER2.from_pretrained("fastino/gliner2-base-v1")
        model.to(DEVICE)

# Evaluate on the test set (all labels)
print("\n" + "="*60)
print("EVALUATING ON TEST SET")
print("="*60)
eval_stats = evaluate(model, test_examples, TARGET_LABELS)

# Save and download the model (using the path that was actually loaded)
# We'll zip the model directory that was used
if os.path.exists(best_continued):
    model_path = best_continued
elif os.path.exists(original_best):
    model_path = original_best
else:
    # If using base model, save it temporarily
    model_path = "./gliner2_base_model"
    os.makedirs(model_path, exist_ok=True)
    model.save_pretrained(model_path)

# Create zip archive
zip_name = os.path.basename(model_path) + ".zip"
zip_path = shutil.make_archive(zip_name.replace(".zip", ""), "zip", model_path)
print(f"\n✅ Model zipped to: {zip_path}")

# Download the zip file (Colab only)
try:
    from google.colab import files
    files.download(zip_path)
    print("📥 Download started.")
except ImportError:
    print("ℹ️ Not in Colab – you can find the zip file at:", zip_path)

[transformers] You are using a model of type `extractor` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


✅ Loading continued fine‑tuned model from ./gliner2_event_boosted/best
🧠 Model Configuration
Encoder model      : microsoft/deberta-v3-base
Counting layer     : count_lstm_v2
Token pooling      : first
Model loaded from continued training.

EVALUATING ON TEST SET
label      precision    recall        f1   support
PERSON         0.910     0.945     0.927      1961
ORG            0.815     0.850     0.832      1751
GPE            0.948     0.951     0.950      2142
EVENT          0.439     0.705     0.541        61
MONEY          0.882     0.907     0.894       313

macro F1: 0.829

✅ Model zipped to: /content/best.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 Download started.


In [5]:
!find /content/drive/MyDrive -type f \( -name "*.zip" -o -name "*.pt" -o -name "*.bin" -o -name "*.safetensors" \) 2>/dev/null | head -100